In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/Users/nitesh/Downloads/customer_shopping_data.csv')
df

,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,05-08-2022,Kanyon
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,12-12-2021,Forum Istanbul
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,09-11-2021,Metrocity
3,I173702,C988172,Female,66,Shoes,5,3000.85,Credit Card,16-05-2021,Metropol AVM
4,I337046,C189076,Female,53,Books,4,60.60,Cash,24-10-2021,Kanyon
...,...,...,...,...,...,...,...,...,...,...
99452,I219422,C441542,Female,45,Souvenir,5,58.65,Credit Card,21-09-2022,Kanyon
99453,I325143,C569580,Male,27,Food & Beverage,2,10.46,Cash,22-09-2021,Forum Istanbul
99454,I824010,C103292,Male,63,Food & Beverage,2,10.46,Debit Card,28-03-2021,Metrocity
99455,I702964,C800631,Male,56,Technology,4,4200.00,Cash,16-03-2021,Istinye Park


In [3]:
df['invoice_date'] = pd.to_datetime(df['invoice_date'], format = 'mixed')
df

,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,2022-05-08,Kanyon
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,2021-12-12,Forum Istanbul
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,2021-09-11,Metrocity
3,I173702,C988172,Female,66,Shoes,5,3000.85,Credit Card,2021-05-16,Metropol AVM
4,I337046,C189076,Female,53,Books,4,60.60,Cash,2021-10-24,Kanyon
...,...,...,...,...,...,...,...,...,...,...
99452,I219422,C441542,Female,45,Souvenir,5,58.65,Credit Card,2022-09-21,Kanyon
99453,I325143,C569580,Male,27,Food & Beverage,2,10.46,Cash,2021-09-22,Forum Istanbul
99454,I824010,C103292,Male,63,Food & Beverage,2,10.46,Debit Card,2021-03-28,Metrocity
99455,I702964,C800631,Male,56,Technology,4,4200.00,Cash,2021-03-16,Istinye Park


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99457 entries, 0 to 99456
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   invoice_no      99457 non-null  object        
 1   customer_id     99457 non-null  object        
 2   gender          99457 non-null  object        
 3   age             99457 non-null  int64         
 4   category        99457 non-null  object        
 5   quantity        99457 non-null  int64         
 6   price           99457 non-null  float64       
 7   payment_method  99457 non-null  object        
 8   invoice_date    99457 non-null  datetime64[ns]
 9   shopping_mall   99457 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(6)
memory usage: 7.6+ MB


In [5]:
df['invoice_date'] = df['invoice_date'].dt.date

In [6]:
df['invoice_date'].max() #max date
#for this problem statement lets consider todays date is (2023,12,02)

datetime.date(2023, 12, 2)

In [7]:
df['Recencey'] = df['invoice_date'].max() - df['invoice_date']    # Recencey
df['Recencey'] = pd.to_timedelta(df['Recencey'])
df['Recencey'] = df['Recencey'].dt.days

In [8]:
df.customer_id.nunique()                                          # Unique Customers 

99457

In [9]:
df['ShopValue'] = df.quantity*df.price                            # ShopValue (Total Amount)

In [10]:
df.head()

,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall,Recencey,ShopValue
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,2022-05-08,Kanyon,573,7502.00
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,2021-12-12,Forum Istanbul,720,5401.53
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,2021-09-11,Metrocity,812,300.08
3,I173702,C988172,Female,66,Shoes,5,3000.85,Credit Card,2021-05-16,Metropol AVM,930,15004.25
4,I337046,C189076,Female,53,Books,4,60.60,Cash,2021-10-24,Kanyon,769,242.40


In [11]:
RFM             = pd.DataFrame()                                     # Create empty DataFrame to store RFM metrics
RFM['Recencey'] = df.groupby(['customer_id']).Recencey.min()         # Recency: most recent purchase per customer
RFM['Freq']     = df.groupby(['customer_id'])['customer_id'].count() # Frequency: number of purchases per customer
RFM['Monetary'] = df.groupby(['customer_id']).ShopValue.sum()        # Monetary: total amount spent per customer

In [12]:
RFM.head(5)

,Recencey,Freq,Monetary
customer_id,,,
C100004,736,1,7502.00
C100005,274,1,2400.68
C100006,689,1,322.56
C100012,839,1,130.75
C100019,860,1,35.84


In [13]:
RFM = RFM.sort_values(['Recencey', 'Freq', 'Monetary'], ascending=[True, False, False])

In [14]:
RFM = RFM.reset_index()

In [15]:
RFM['Recencey_Rank'] = RFM['Recencey'].rank(ascending= False) # Recent buyers get better rank
RFM['Monetary_Rank'] = RFM['Monetary'].rank(ascending= True)  # Higher spending customers
RFM['Freq_Rank'] = RFM['Freq'].rank(ascending= True)          # Customers who purchase more often

In [16]:
RFM.head()

,customer_id,Recencey,Freq,Monetary,Recencey_Rank,Monetary_Rank,Freq_Rank
0,C376513,0,1,15004.25,99393.0,96449.0,49729.0
1,C106686,0,1,9602.72,99393.0,94431.5,49729.0
2,C148206,0,1,9602.72,99393.0,94431.5,49729.0
3,C163422,0,1,9602.72,99393.0,94431.5,49729.0
4,C177544,0,1,9602.72,99393.0,94431.5,49729.0


In [17]:
RFM['Rece_Rank_Range'] = RFM['Recencey_Rank']/RFM['Recencey_Rank'].max() # Normalize Recency rank to a 0–1 scale
RFM['mone_Rank_Range'] = RFM['Monetary_Rank']/RFM['Monetary_Rank'].max() # Normalize Monetary rank to a 0–1 scale
RFM['freq_Rank_Range'] = RFM['Freq_Rank']/RFM['Freq_Rank'].max()         # Normalize Frequency rank to a 0–1 scale

In [18]:
RFM

,customer_id,Recencey,Freq,Monetary,Recencey_Rank,Monetary_Rank,Freq_Rank,Rece_Rank_Range,mone_Rank_Range,freq_Rank_Range
0,C376513,0,1,15004.25,99393.0,96449.0,49729.0,1.000000,0.974656,1.0
1,C106686,0,1,9602.72,99393.0,94431.5,49729.0,1.000000,0.954268,1.0
2,C148206,0,1,9602.72,99393.0,94431.5,49729.0,1.000000,0.954268,1.0
3,C163422,0,1,9602.72,99393.0,94431.5,49729.0,1.000000,0.954268,1.0
4,C177544,0,1,9602.72,99393.0,94431.5,49729.0,1.000000,0.954268,1.0
...,...,...,...,...,...,...,...,...,...,...
99452,C210302,1065,1,20.92,53.0,6466.0,49729.0,0.000533,0.065342,1.0
99453,C316581,1065,1,20.92,53.0,6466.0,49729.0,0.000533,0.065342,1.0
99454,C339057,1065,1,20.92,53.0,6466.0,49729.0,0.000533,0.065342,1.0
99455,C535517,1065,1,11.73,53.0,3514.5,49729.0,0.000533,0.035515,1.0


In [19]:
RFM['Avg_Rank'] = (RFM['Rece_Rank_Range'] + RFM['mone_Rank_Range'] + RFM['freq_Rank_Range']) / 3 
# Compute overall RFM score by averaging normalized Recency, Monetary, and Frequency ranks

In [20]:
RFM

,customer_id,Recencey,Freq,Monetary,Recencey_Rank,Monetary_Rank,Freq_Rank,Rece_Rank_Range,mone_Rank_Range,freq_Rank_Range,Avg_Rank
0,C376513,0,1,15004.25,99393.0,96449.0,49729.0,1.000000,0.974656,1.0,0.991552
1,C106686,0,1,9602.72,99393.0,94431.5,49729.0,1.000000,0.954268,1.0,0.984756
2,C148206,0,1,9602.72,99393.0,94431.5,49729.0,1.000000,0.954268,1.0,0.984756
3,C163422,0,1,9602.72,99393.0,94431.5,49729.0,1.000000,0.954268,1.0,0.984756
4,C177544,0,1,9602.72,99393.0,94431.5,49729.0,1.000000,0.954268,1.0,0.984756
...,...,...,...,...,...,...,...,...,...,...,...
99452,C210302,1065,1,20.92,53.0,6466.0,49729.0,0.000533,0.065342,1.0,0.355292
99453,C316581,1065,1,20.92,53.0,6466.0,49729.0,0.000533,0.065342,1.0,0.355292
99454,C339057,1065,1,20.92,53.0,6466.0,49729.0,0.000533,0.065342,1.0,0.355292
99455,C535517,1065,1,11.73,53.0,3514.5,49729.0,0.000533,0.035515,1.0,0.345350


In [21]:
RFM[RFM['customer_id'] == 'C111359'] 
# Filter RFM table to view metrics for a specific customer

,customer_id,Recencey,Freq,Monetary,Recencey_Rank,Monetary_Rank,Freq_Rank,Rece_Rank_Range,mone_Rank_Range,freq_Rank_Range,Avg_Rank
121,C111359,0,1,5.23,99393.0,1501.5,49729.0,1.0,0.015173,1.0,0.671724
